<a href="https://colab.research.google.com/github/sathu0622/25-26J-438-AI-Powered-LMS-for-Visually-Impaired-Students/blob/AI-Powered-Braille-to-Text-Conversion-and-Automated-Evaluation-System-for-O%2FL-History-Examinations/CNN_Approch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# ✅ FULL BRAILLE CNN (No Pruning) + Dataset & Models from Google Drive
# Fixed: Proper Drive mounting with authorization
# ==========================================

import os
import numpy as np
import cv2
import tensorflow as tf
from tensorflow import keras
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# ==========================================
# 0. MOUNT GOOGLE DRIVE
# ==========================================

try:
    from google.colab import drive

    # Mount with force_remount to ensure fresh authorization
    drive.mount('/content/drive', force_remount=True)

    drive_path = "/content/drive/MyDrive/braille"  # folder to save models
    dataset_path = "/content/drive/MyDrive/braille/Braille_Dataset"  # folder where dataset is stored

    # Create directory if it doesn't exist
    os.makedirs(drive_path, exist_ok=True)

    print("✅ Google Drive mounted successfully!")
    print(f"Drive path: {drive_path}")
    print(f"Dataset path: {dataset_path}")

except Exception as e:
    print(f"❌ Error mounting Google Drive: {e}")
    print("\nPlease ensure:")
    print("1. You're running this in Google Colab")
    print("2. You have authorized Drive access when prompted")
    print("3. The dataset folder exists at the specified path")
    raise

# ==========================================
# 1. LOAD DATASET IMAGES
# ==========================================

image_dir = Path(dataset_path)

# Check if dataset path exists
if not image_dir.exists():
    raise FileNotFoundError(f"Dataset path does not exist: {dataset_path}")

dir_list = list(image_dir.rglob("*.jpg"))
print(f"Total Images Found: {len(dir_list)}")

if len(dir_list) == 0:
    raise ValueError("No .jpg images found in the dataset path. Please check your dataset location.")

# ==========================================
# 2. LOAD IMAGES + LABELS
# ==========================================

images = []
labels = []

print("Loading images...")
for img_path in dir_list:
    labels.append(img_path.name[0])  # first character = label
    img = cv2.imread(str(img_path))

    if img is None:
        print(f"Warning: Could not read image {img_path}")
        continue

    img = cv2.resize(img, (64, 64))   # resize all images
    images.append(img)

images = np.array(images) / 255.0
labels = np.array(labels)

print(f"Loaded {len(images)} images successfully")

# Encode labels A-Z → 0–25
le = LabelEncoder()
labels = le.fit_transform(labels)

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    images, labels, test_size=0.2, random_state=42
)
print("Training Shape:", X_train.shape)
print("Testing Shape :", X_test.shape)

# ==========================================
# 3. BUILD CNN MODEL
# ==========================================

model = keras.Sequential([
    keras.layers.Conv2D(64, (5,5), activation="relu", padding="same", input_shape=(64,64,3)),
    keras.layers.MaxPooling2D(),

    keras.layers.Conv2D(64, (3,3), activation="relu", padding="same"),
    keras.layers.MaxPooling2D(),
    keras.layers.Dropout(0.25),

    keras.layers.Conv2D(64, (3,3), activation="relu", padding="same"),
    keras.layers.MaxPooling2D(),
    keras.layers.Dropout(0.25),

    keras.layers.Flatten(),

    keras.layers.Dense(256, activation="relu"),
    keras.layers.Dropout(0.3),

    keras.layers.Dense(26, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

# ==========================================
# 4. TRAIN ORIGINAL MODEL
# ==========================================

print("\n🚀 Training CNN...\n")

history = model.fit(
    X_train, y_train,
    epochs=15,
    validation_split=0.2,
    batch_size=32
)

# Evaluate Original Model
orig_loss, orig_acc = model.evaluate(X_test, y_test)
print("\n✅ Original Model Accuracy:", round(orig_acc * 100, 2), "%")

# Save Original Model to Google Drive
model_path = os.path.join(drive_path, "cnn_braille.keras")
model.save(model_path)
print("Original Model Saved in Drive:", model_path)
print("Model Size:", round(os.path.getsize(model_path) / 1024, 2), "KB")

# ==========================================
# 5. QUANTIZATION (TFLite)
# ==========================================

print("\n⚡ Applying Quantization...\n")

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

quantized_tflite_model = converter.convert()

tflite_path = os.path.join(drive_path, "braille_quantized.tflite")
with open(tflite_path, "wb") as f:
    f.write(quantized_tflite_model)

print("Quantized TFLite Model Saved in Drive:", tflite_path)
print("Quantized Model Size:", round(os.path.getsize(tflite_path) / 1024, 2), "KB")

# ==========================================
# 6. EVALUATE TFLITE MODEL ACCURACY
# ==========================================

print("\n📌 Evaluating Quantized TFLite Model...\n")

interpreter = tf.lite.Interpreter(model_path=tflite_path)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

correct = 0
for i in range(len(X_test)):
    input_data = np.expand_dims(X_test[i], axis=0).astype(np.float32)
    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()
    output = interpreter.get_tensor(output_details[0]['index'])
    pred = np.argmax(output)
    if pred == y_test[i]:
        correct += 1

tflite_acc = correct / len(X_test)
print("✅ Quantized TFLite Accuracy:", round(tflite_acc * 100, 2), "%")

# ==========================================
# 7. FINAL SUMMARY
# ==========================================

print("\n===================================")
print("📌 FINAL ACCURACY RESULTS")
print("===================================")
print("Original Model Accuracy :", round(orig_acc * 100, 2), "%")
print("TFLite Model Accuracy   :", round(tflite_acc * 100, 2), "%")
print("===================================")

print("\n🎉 DONE SUCCESSFULLY! All files saved to Drive.")
print(f"\nFiles saved to: {drive_path}")
print("- cnn_braille.keras")
print("- braille_quantized.tflite")

Mounted at /content/drive
✅ Google Drive mounted successfully!
Drive path: /content/drive/MyDrive/braille
Dataset path: /content/drive/MyDrive/braille/Braille_Dataset
Total Images Found: 1560
Loading images...
Loaded 1560 images successfully
Training Shape: (1248, 64, 64, 3)
Testing Shape : (312, 64, 64, 3)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 64, 64, 64)     │         4,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 32, 32, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 16, 16, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     1,048,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 26)             │         6,682 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,134,234 (4.33 MB)

 Trainable params: 1,134,234 (4.33 MB)

 Non-trainable params: 0 (0.00 B)


🚀 Training CNN...

Epoch 1/15
32/32 ━━━━━━━━━━━━━━━━━━━━ 19s 515ms/step - accuracy: 0.0426 - loss: 3.2898 - val_accuracy: 0.0280 - val_loss: 3.2612
Epoch 2/15
32/32 ━━━━━━━━━━━━━━━━━━━━ 18s 557ms/step - accuracy: 0.0540 - loss: 3.2274 - val_accuracy: 0.3040 - val_loss: 2.5875
Epoch 3/15
32/32 ━━━━━━━━━━━━━━━━━━━━ 18s 457ms/step - accuracy: 0.3291 - loss: 2.4091 - val_accuracy: 0.6600 - val_loss: 1.4649
Epoch 4/15
32/32 ━━━━━━━━━━━━━━━━━━━━ 15s 462ms/step - accuracy: 0.5886 - loss: 1.4385 - val_accuracy: 0.7000 - val_loss: 1.1015
Epoch 5/15
32/32 ━━━━━━━━━━━━━━━━━━━━ 20s 468ms/step - accuracy: 0.6931 - loss: 1.0201 - val_accuracy: 0.7400 - val_loss: 1.0063
Epoch 6/15
32/32 ━━━━━━━━━━━━━━━━━━━━ 15s 465ms/step - accuracy: 0.7798 - loss: 0.7536 - val_accuracy: 0.7560 - val_loss: 0.9528
Epoch 7/15
32/32 ━━━━━━━━━━━━━━━━━━━━ 15s 483ms/step - accuracy: 0.8349 - loss: 0.5252 - val_accuracy: 0.7920 - val_loss: 0.9801
Epoch 8/15
32/32 ━━━━━━━━━━━━━━━━━━━━ 15s 477ms/step - accuracy: 0.8635 - los

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


✅ Quantized TFLite Accuracy: 83.97 %

📌 FINAL ACCURACY RESULTS
Original Model Accuracy : 83.65 %
TFLite Model Accuracy   : 83.97 %

🎉 DONE SUCCESSFULLY! All files saved to Drive.

Files saved to: /content/drive/MyDrive/braille
- cnn_braille.keras
- braille_quantized.tflite
